In [1]:
# Original sketch for pipeline

from common import *
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from tqdm.auto import tqdm

from IPDiff.utils.visualize import visualize_protein_ligand
import IPDiff.utils.transforms as trans
from IPDiff.utils import reconstruct
from IPDiff.utils import misc

load_path = root_dir + '/sampled_results/7upg_pocket1'
protein_path = root_dir + '/pockets/7upg_pocket1.pdb'
result_files = [f for f in os.listdir(load_path) if f.endswith('.pt')]

# Inspect diffusion results
for result_file in result_files:
    print(result_file)

/home/adrianchen/miniconda3/envs/protein-docking/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sample_2025-03-20_22-57-38_079.pt
sample_2025-03-22_03-55-49.pt
sample_2025-04-11_01-32-22_039.pt


In [2]:
result = torch.load(os.path.join(load_path, result_files[2]))
pred_ligand_pos, pred_ligand_v = [], []
for sample_idx, (pred_pos_batch, pred_v_batch) in enumerate(zip(result['pred_ligand_pos'], result['pred_ligand_v'])):
    pred_ligand_pos += pred_pos_batch
    pred_ligand_v += pred_v_batch
print("Number of generated ligands:", len(pred_ligand_pos))

Number of generated ligands: 1000


In [4]:
os.makedirs('temp', exist_ok=True)

ligand_pred = []
reconstruct_failed, smiles_failed = [], []
for sample_idx, (pred_pos, pred_v) in tqdm(enumerate(zip(pred_ligand_pos, pred_ligand_v)), total=len(pred_ligand_pos)):
    pred_atom_type = trans.get_atomic_number_from_index(pred_v, mode='add_aromatic')
    try:
        pred_aromatic = trans.is_aromatic_from_index(pred_v, mode='add_aromatic')
        mol = reconstruct.reconstruct_from_generated(pred_pos, pred_atom_type, pred_aromatic)
        smiles = Chem.MolToSmiles(mol)
    except reconstruct.MolReconsError:
        reconstruct_failed.append(sample_idx)
        continue
    
    if '.' in smiles:
        smiles_failed.append(smiles)
        continue
    
    ligand_pred.append(mol)
    sdf_writer = Chem.SDWriter(os.path.join('temp', f'{sample_idx:03d}.sdf'))
    sdf_writer.write(mol)
    sdf_writer.close()

print("Successfully reconstructed", len(ligand_pred), "ligands")
print("Failed to reconstruct", len(reconstruct_failed), "ligands")
print("Failed when getting smiles", len(smiles_failed), "ligands")

100%|██████████| 1000/1000 [00:35<00:00, 28.56it/s]

Successfully reconstructed 874 ligands
Failed to reconstruct 13 ligands
Failed when getting smiles 113 ligands


In [5]:
for idx in range(10):
    mol = ligand_pred[idx]
    if protein_path is None:
        protein_path = os.path.join(train_config.data.path, result['data']['protein_filename'].split('rec')[0] + 'rec.pdb')
    with open(protein_path, 'r') as f:
        pdb_block = f.read()
    sdf_block = Chem.MolToMolBlock(mol)

    vis = visualize_protein_ligand(pdb_block, sdf_block, show_ligand=True, show_surface=True)
    vis.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
# read ZINC library
ZINC_path = '/local/adrianchen/ZINC20-drug'
ZINC_lib_master_tranches = [_ for _ in os.listdir(ZINC_path) if os.path.isdir(os.path.join(ZINC_path, _))]
ZINC_lib_master_tranche_dict = {
    dm: [_ for _ in os.listdir(os.path.join(ZINC_path, dm)) if os.path.isdir(os.path.join(ZINC_path, dm, _))]
    for dm in ZINC_lib_master_tranches
}
ZINC_lib_all_tranche_paths = [
    os.path.join(ZINC_path, dm, lib)
    for dm in ZINC_lib_master_tranches
    for lib in ZINC_lib_master_tranche_dict[dm]
]
ZINC_lib_all_tranche_dict = {
    master_tranche: {
        sub_tranche: [_ for _ in os.listdir(os.path.join(ZINC_path, master_tranche, sub_tranche))
                      if os.path.isdir(os.path.join(ZINC_path, master_tranche, sub_tranche, _))]
        for sub_tranche in ZINC_lib_master_tranche_dict[master_tranche]
    }
    for master_tranche in ZINC_lib_master_tranches
}
ZINC_lib_all_sdf_paths = [
    os.path.join(tranche, sdf_file)
    for tranche in ZINC_lib_all_tranche_paths
    for sdf_file in os.listdir(tranche)
    if sdf_file.endswith('.sdf.gz')
]
print(f"Done exploring ZINC library, total tranches: {len(ZINC_lib_all_tranche_paths)}, total sdf files: {len(ZINC_lib_all_sdf_paths)}")

Done exploring ZINC library, total tranches: 716, total sdf files: 758


In [15]:
from postprocess_results import find_tranche, read_molecules_from_sdf

master_tranche = find_tranche(ligand_pred[1])
sub_tranches = ZINC_lib_master_tranche_dict[master_tranche]
for idx_sub_tranche, sub_tranche in enumerate(sub_tranches):
    sub_tranche_path = os.path.join(ZINC_path, master_tranche, sub_tranche)
    sdf_files = os.listdir(sub_tranche_path)
    sdf_files = [os.path.join(sub_tranche_path, sdf_file) for sdf_file in sdf_files if sdf_file.endswith('.sdf.gz')]
    molecules = read_molecules_from_sdf(sdf_files)
    
    print(idx_sub_tranche)
    if idx_sub_tranche > 10:
        break

[06:09:52] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:52] ERROR: Could not sanitize molecule ending on line 42816
[06:09:52] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:52] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:52] ERROR: Could not sanitize molecule ending on line 45043
[06:09:52] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:53] Explicit valence for atom # 8 N, 4, is greater than permitted
[06:09:53] ERROR: Could not sanitize molecule ending on line 147866
[06:09:53] ERROR: Explicit valence for atom # 8 N, 4, is greater than permitted
[06:09:53] Explicit valence for atom # 8 N, 4, is greater than permitted
[06:09:53] ERROR: Could not sanitize molecule ending on line 148028
[06:09:53] ERROR: Explicit valence for atom # 8 N, 4, is greater than permitted
[06:09:53] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:53] ERROR: Could not sanitize molecule

0
1


[06:09:53] Explicit valence for atom # 10 N, 4, is greater than permitted
[06:09:53] ERROR: Could not sanitize molecule ending on line 1446
[06:09:53] ERROR: Explicit valence for atom # 10 N, 4, is greater than permitted


2


[06:09:55] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:55] ERROR: Could not sanitize molecule ending on line 217649
[06:09:55] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:55] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:55] ERROR: Could not sanitize molecule ending on line 217920
[06:09:55] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:55] Explicit valence for atom # 20 N, 4, is greater than permitted
[06:09:55] ERROR: Could not sanitize molecule ending on line 358612
[06:09:55] ERROR: Explicit valence for atom # 20 N, 4, is greater than permitted
[06:09:57] Explicit valence for atom # 11 N, 4, is greater than permitted
[06:09:57] ERROR: Could not sanitize molecule ending on line 640245
[06:09:57] ERROR: Explicit valence for atom # 11 N, 4, is greater than permitted
[06:09:58] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:09:58] ERROR: Could not sanitize mo

...

4
5


[06:09:59] Explicit valence for atom # 6 N, 4, is greater than permitted
[06:09:59] ERROR: Could not sanitize molecule ending on line 1502
[06:09:59] ERROR: Explicit valence for atom # 6 N, 4, is greater than permitted
[06:09:59] Explicit valence for atom # 9 N, 4, is greater than permitted
[06:09:59] ERROR: Could not sanitize molecule ending on line 19555
[06:09:59] ERROR: Explicit valence for atom # 9 N, 4, is greater than permitted
[06:09:59] Explicit valence for atom # 6 N, 4, is greater than permitted
[06:09:59] ERROR: Could not sanitize molecule ending on line 19622
[06:09:59] ERROR: Explicit valence for atom # 6 N, 4, is greater than permitted
[06:09:59] Explicit valence for atom # 3 N, 4, is greater than permitted
[06:09:59] ERROR: Could not sanitize molecule ending on line 13366
[06:09:59] ERROR: Explicit valence for atom # 3 N, 4, is greater than permitted
[06:09:59] Explicit valence for atom # 20 N, 4, is greater than permitted
[06:09:59] ERROR: Could not sanitize molecule e

6


[06:10:00] Explicit valence for atom # 18 N, 4, is greater than permitted
[06:10:00] ERROR: Could not sanitize molecule ending on line 201016
[06:10:00] ERROR: Explicit valence for atom # 18 N, 4, is greater than permitted
[06:10:03] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:03] ERROR: Could not sanitize molecule ending on line 861624
[06:10:03] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:03] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:03] ERROR: Could not sanitize molecule ending on line 862392
[06:10:03] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:04] Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:04] ERROR: Could not sanitize molecule ending on line 1091352
[06:10:04] ERROR: Explicit valence for atom # 1 N, 4, is greater than permitted
[06:10:09] Explicit valence for atom # 16 N, 4, is greater than permitted
[06:10:09] ERROR: Could not sanitize mo

7


In [1]:
# Hide warnings
import warnings
warnings.filterwarnings('ignore')

def get_similarity(mol1, mol2):
    from rdkit.Chem import rdFingerprintGenerator
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    fp1 = gen.GetFingerprint(mol1)
    fp2 = gen.GetFingerprint(mol2)

    return DataStructs.FingerprintSimilarity(fp1, fp2)

def get_top_k_similar_mols(mol, ZINC_mols, k=10):
    similarities = []
    for key, mol2 in ZINC_mols.items():
        similarity = get_similarity(mol, mol2)
        similarities.append((similarity, key))
    similarities.sort(key=lambda x: x[0], reverse=True)
    return similarities[:k]

def visualize_similar_mols(mol1, mol2):
    # compute 3D coordinates
    pos1, pos2 = mol1.GetConformer().GetPositions(), mol2.GetConformer().GetPositions()
    
    # centralize both molecules
    pos1 -= pos1.mean(axis=0)
    pos2 -= pos2.mean(axis=0)
    
    # pca to rotate both molecules to the same direction
    from sklearn.decomposition import PCA
    pca = PCA(n_components=3)
    pca.fit(pos1)
    pos1 = pos1 @ pca.components_.T
    pca.fit(pos2)
    pos2 = pos2 @ pca.components_.T
    
    # create offset along the second principal component
    dist = np.max(pos1, axis=0) - np.min(pos2, axis=0)
    offset = np.array([0, dist[1] + 3, 0])
    pos2 += offset / 2
    pos1 -= offset / 2
    
    mol1.GetConformer().SetPositions(pos1)
    mol2.GetConformer().SetPositions(pos2)
    
    import py3Dmol
    view = py3Dmol.view()
    view.addModel(Chem.MolToMolBlock(mol1), 'sdf')
    view.setStyle({'model': -1}, {'stick': {}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.8}, {'model': -1})
    view.addModel(Chem.MolToMolBlock(mol2), 'sdf')
    view.setStyle({'model': -1}, {'stick': {}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.8}, {'model': -1})
    view.show()

similarity_threshold = 0.22
candidates = []
visualize = True
for idx_ligand, ligand in enumerate(ligand_pred):
    mol1 = ligand
    similarities = get_top_k_similar_mols(mol1, ZINC_mols, k=10)
    for similarity, key in similarities:
        if similarity > similarity_threshold:
            mol2 = ZINC_mols[key]
            candidates.append((similarity, key))
            print(idx_ligand, key, similarity)
            if visualize:
                visualize_similar_mols(mol1, mol2)

NameError: name 'ligand_pred' is not defined

In [ ]:
mol = list(ZINC_mols.values())[2]
similarities = get_top_k_similar_mols(mol, ZINC_mols, k=10)
print(similarities)
visualize_similar_mols(ZINC_mols[similarities[1][1]], mol)

In [ ]:
from IPDiff.utils.evaluation.docking_vina import VinaDockingTask

import openbabel
def convert_pdbqt_to_pdb(pdbqt_file, pdb_file):
    obConversion = openbabel.OBConversion()
    # Set input and output formats
    obConversion.SetInAndOutFormats("pdbqt", "pdb")
    
    mol = openbabel.OBMol()
    if obConversion.ReadFile(mol, pdbqt_file):
        obConversion.WriteFile(mol, pdb_file)
    else:
        print("Failed to read the PDBQT file.")

with open(protein_path, 'r') as f:
    pdb_block = f.read()

for candidate in candidates:
    similarity, key = candidate
    mol = ZINC_mols[key]
    vina_task = VinaDockingTask(protein_path, mol)
    docking_results = vina_task.run(exhaustiveness=32)
    affinity, pose = docking_results[0]['affinity'], docking_results[0]['pose']
    print(key, affinity)
    
    # Convert PDBQT pose to PDB format
    os.makedirs('tmp', exist_ok=True)
    pdbqt_file = os.path.join('tmp', f'{key}.pdbqt')
    pdb_file = os.path.join('tmp', f'{key}.pdb')
    with open(pdbqt_file, 'w') as f:
        f.write(pose)
    convert_pdbqt_to_pdb(pdbqt_file, pdb_file)
    pdb_ligand = open(pdb_file, 'r').read()
    
    if affinity > -10:
        try:
            # Convert vina pose to sdf block
            mol = Chem.MolFromPDBBlock(pdb_ligand)
            sdf_block = Chem.MolToMolBlock(mol)
            visualize_protein_ligand(pdb_block, sdf_block, show_ligand=True, show_surface=True).show()
        except Exception as e:
            print(f"Error processing {key}: {e}")